In [0]:
%pip install -q pytest chispa

In [0]:
import os
import sys
from datetime import datetime, timezone
from pathlib import Path

import pytest

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
project_root = Path("/Workspace") / notebook_path.lstrip("/")
project_root = project_root.parent

# Databricks Workspace/Repos може обмежувати службові cache files.
sys.dont_write_bytecode = True

os.chdir(project_root)
sys.path.insert(0, str(project_root / "src"))

SUMMARY_TABLE = "pyspark_demo_test_run_results"
DETAILS_TABLE = "pyspark_demo_test_case_results"


class PytestRunCollector:
    def __init__(self):
        self.reports = []

    def pytest_runtest_logreport(self, report):
        if report.when != "call":
            return

        self.reports.append(
            {
                "test_name": report.nodeid,
                "outcome": report.outcome,
                "duration_seconds": float(report.duration),
            }
        )


collector = PytestRunCollector()
started_at = datetime.now(timezone.utc)
run_id = started_at.strftime("%Y%m%dT%H%M%S%fZ")

exit_code = pytest.main([
    "tests",
    "-v",
    "-m", "not local_only",
    "-p", "no:cacheprovider",
], plugins=[collector])

finished_at = datetime.now(timezone.utc)
total = len(collector.reports)
passed = sum(1 for report in collector.reports if report["outcome"] == "passed")
failed = sum(1 for report in collector.reports if report["outcome"] == "failed")
skipped = sum(1 for report in collector.reports if report["outcome"] == "skipped")
status = "PASSED" if int(exit_code) == 0 else "FAILED"

summary_rows = [(
    run_id,
    started_at,
    finished_at,
    float((finished_at - started_at).total_seconds()),
    status,
    int(exit_code),
    total,
    passed,
    failed,
    skipped,
    str(project_root),
)]

summary_schema = "run_id string, started_at timestamp, finished_at timestamp, duration_seconds double, status string, exit_code int, total_tests int, passed_tests int, failed_tests int, skipped_tests int, project_root string"
spark.createDataFrame(summary_rows, summary_schema).write.mode("append").format("delta").saveAsTable(SUMMARY_TABLE)

detail_rows = [
    (run_id, report["test_name"], report["outcome"], report["duration_seconds"])
    for report in collector.reports
]
detail_schema = "run_id string, test_name string, outcome string, duration_seconds double"
spark.createDataFrame(detail_rows, detail_schema).write.mode("append").format("delta").saveAsTable(DETAILS_TABLE)

display(spark.table(SUMMARY_TABLE).where(f"run_id = '{run_id}'"))

if int(exit_code) != 0:
    raise AssertionError(f"pytest failed with exit code {int(exit_code)}")